In [ ]:
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    print("Downloading NLTK data...")
    nltk.download('stopwords')

french_stop_words = stopwords.words('french')
my_extra_words = [
    'incident', 'client', 'signale', 'demande', 'reclamation', 'bonjour', 
    'assistance', 'problème', 'rapporte', 'réalisée', 'réclamation', 'service',
    'certaines', 'continue', 'depuis', 'important', 'lors', 'matin', 
    'non', 'soirée', 'très', 'merci', 'cordialement'
]
french_stop_words.extend(my_extra_words)

try:
    df = pd.read_csv('algerietelecom.csv')
    print("Data Loaded Successfully.")
except FileNotFoundError:
    print("Error: 'algerietelecom.csv' not found.")
    exit()

df['urgence'] = df['urgence'].str.capitalize()
df['sentiment'] = df['sentiment'].str.capitalize()
df['categorie'] = df['categorie'].str.capitalize()
df['wilaya'] = df['wilaya'].str.capitalize()
if 'status' in df.columns:
    df['status'] = df['status'].str.capitalize()

sns.set_style("whitegrid")
fig, axes = plt.subplots(4, 2, figsize=(18, 20))

sns.countplot(y='categorie', data=df, order=df['categorie'].value_counts().index, ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Distribution by Category')
axes[0, 0].set_xlabel('Count')

sns.countplot(x='canal', data=df, order=df['canal'].value_counts().index, ax=axes[0, 1], palette='magma')
axes[0, 1].set_title('Submission Channels')
axes[0, 1].set_xlabel('Channel')

sns.countplot(x='urgence', data=df, order=['Basse', 'Moyenne', 'Élevée', 'Critique'], ax=axes[1, 0], palette='Reds')
axes[1, 0].set_title('Urgency Levels')
axes[1, 0].set_xlabel('Urgency')

sns.countplot(x='sentiment', data=df, order=['Négatif', 'Neutre', 'Positif'], ax=axes[1, 1], palette='coolwarm')
axes[1, 1].set_title('Sentiment Analysis')
axes[1, 1].set_xlabel('Sentiment')

if 'status' in df.columns:
    sns.countplot(x='status', data=df, order=df['status'].value_counts().index, ax=axes[2, 0], palette='Greens')
    axes[2, 0].set_title('Complaint Status')
    axes[2, 0].set_xlabel('Status')
else:
    axes[2, 0].axis('off')

if 'suggested_action' in df.columns:
    sns.countplot(y='suggested_action', data=df, order=df['suggested_action'].value_counts().index, ax=axes[2, 1], palette='Oranges')
    axes[2, 1].set_title('Suggested Actions (Current Labels)')
    axes[2, 1].set_xlabel('Count')
else:
    axes[2, 1].axis('off')

top_wilayas = df['wilaya'].value_counts().head(10).index
sns.countplot(y='wilaya', data=df[df['wilaya'].isin(top_wilayas)], order=top_wilayas, ax=axes[3, 0], palette='Blues_r')
axes[3, 0].set_title('Top 10 Wilayas')
axes[3, 0].set_xlabel('Count')

axes[3, 1].axis('off')

plt.tight_layout()
plt.show()

cv = CountVectorizer(stop_words=french_stop_words, max_features=15, ngram_range=(1, 2))
X_counts = cv.fit_transform(df['texte_reclamation'].astype(str))
count_sum = X_counts.sum(axis=0)
words_freq = [(word, count_sum[0, idx]) for word, idx in cv.vocabulary_.items()]
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

top_words_df = pd.DataFrame(words_freq, columns=['Word', 'Frequency'])
plt.figure(figsize=(10, 6))
sns.barplot(x='Frequency', y='Word', data=top_words_df, palette='Blues_r')
plt.title('Top 15 Keywords in Complaints')
plt.show()

features_to_use = ['texte_reclamation', 'wilaya', 'canal', 'urgence', 'sentiment', 'categorie']
if 'status' in df.columns:
    features_to_use.append('status')

X = df[features_to_use]

urgence_order = ['Basse', 'Moyenne', 'Élevée', 'Critique']
sentiment_order = ['Négatif', 'Neutre', 'Positif']

nominal_features = ['wilaya', 'canal', 'categorie']
if 'status' in df.columns:
    nominal_features.append('status')

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(stop_words=french_stop_words, max_features=2000), 'texte_reclamation'),
        ('nominal', OneHotEncoder(handle_unknown='ignore'), nominal_features),
        ('ordinal', OrdinalEncoder(categories=[urgence_order, sentiment_order], handle_unknown='use_encoded_value', unknown_value=-1), ['urgence', 'sentiment'])
    ]
)

X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

print(f"Preprocessing Complete for Clustering.")
print(f"Total features created: {X_transformed.shape[1]}")
print("Example Features:", feature_names[:10])